In [1]:
setwd("/mnt/gsdata/projects/panops/panops-data-registry/data/flux")

In [4]:
# --- Cell 1: Setup ---
#install.packages("future.apply")  
library(data.table)
library(future)
library(stringr)
library(future.apply)  # install.packages("future.apply") if needed

# Path to your big ZIP file
zipfile <- "HH_flux.zip"

# Columns you care about
wanted_cols <- c(
  "TIMESTAMP_START", "TIMESTAMP_END",
  "TA_F", "TA_F_QC", "VPD_F", "VPD_F_QC", "P_F", "P_F_QC",
  "TS_F_MDS_1", "TS_F_MDS_2", "TS_F_MDS_1_QC", "TS_F_MDS_2_QC",
  "WS_F", "WS_F_QC",
  "SWC_F_MDS_1", "SWC_F_MDS_2", "SWC_F_MDS_1_QC", "SWC_F_MDS_2_QC",
  "H_CORR", "H_RANDUNC", "H_RANDUNC_N", "NEE_VUT_REF_RANDUNC_N",
  "LE_F_MDS", "LE_F_MDS_QC", "LE_CORR", "LE_RANDUNC", "LE_RANDUNC_N",
  "H_F_MDS", "H_F_MDS_QC", "CO2_F_MDS", "CO2_F_MDS_QC",
  "NEE_VUT_REF", "NEE_VUT_REF_QC", "NEE_VUT_50", "NEE_VUT_50_QC",
  "NEE_VUT_REF_RANDUNC",
  "GPP_NT_VUT_REF", "GPP_DT_VUT_REF", "GPP_NT_VUT_50", "GPP_DT_VUT_50",
  "RECO_NT_VUT_REF", "RECO_DT_VUT_REF", "RECO_NT_VUT_50", "RECO_DT_VUT_50",
  "SW_IN_F", "SW_IN_F_QC", "PPFD_IN", "PPFD_DIF", "PPFD_OUT",
  "TA_F_MDS", "TA_F_MDS_QC", "VPD_F_MDS",
  "SW_IN_F_MDS", "SW_IN_F_MDS_QC", "SW_IN_POT",
  "LW_IN_F", "LW_OUT", "SW_OUT", "NETRAD",
  "USTAR", "WS", "G_F_MDS", "G_F_MDS_QC", "RH", "NIGHT", "PA"
)

# New output folder (subset/clean version)
outdir <- "merged_sites_subset"
dir.create(outdir, showWarnings = FALSE)

# Be polite with threads (important on shared machines)
data.table::setDTthreads(2)

In [5]:
# --- Cell 2: Discover files and site IDs ---

# List all files inside the ZIP (without extracting everything)
zip_files <- unzip(zipfile, list = TRUE)

# Keep only CSVs
csv_files <- zip_files$Name[grepl("\\.csv$", zip_files$Name)]

# Extract site IDs like AR-Bal, US-Ro2, etc.
extract_site <- function(x) {
  str_extract(basename(x), "[A-Z]{2}-[A-Za-z0-9]+")
}

site_ids     <- extract_site(csv_files)
unique_sites <- unique(site_ids[!is.na(site_ids)])

length(unique_sites)
head(unique_sites)

[1] 521

[1] "US-Wi0" "ZA-Kru" "US-xWR" "AU-ASM" "CA-Gro" "JP-BBY"

In [6]:
# --- Cell 3: Robust streaming reader from ZIP ---

# fread wrapper that:
#  - reads from a shell command
#  - allows extra fields: fill = Inf (data.table >= 1.15)
safe_fread_cmd <- function(cmd) {
  fread(cmd = cmd, showProgress = FALSE, fill = Inf)
}

# Read a single CSV *inside* the zip, removing NULs
read_file_from_zip <- function(fname_in_zip) {
  cmd <- sprintf("unzip -p %s %s | tr -d '\\000'",
                 shQuote(zipfile), shQuote(fname_in_zip))
  dt <- safe_fread_cmd(cmd)
  
  # Drop junk Mac OS X / quarantine footer lines if present
  if (nrow(dt) > 0) {
    col1 <- dt[[1]]
    bad  <- grepl("^Mac OS X", col1) | grepl("com\\.apple\\.quarantine", col1)
    if (any(bad)) {
      dt <- dt[!bad]
    }
  }
  
  dt
}

In [6]:
# --- Cell 4: Sites that previously had fread warnings ---

sites_with_warning <- c(
  "NL-Loo", "US-Myb", "US-Wi9", "CA-NS1", "SE-Svb", "FR-Gri", "US-Tw1",
  "IT-Noe", "CA-NS2", "CA-NS4", "FR-Fon", "US-KS2", "US-Oho", "BE-Vie",
  "ES-LMa", "US-Twt", "US-ORv", "US-Whs", "FI-Sod", "CA-TPD", "DE-RuS",
  "IT-BCi", "CA-Qfo", "US-RGo", "US-Wi0", "US-ARM", "CA-TP3", "AU-Cum",
  "SE-Nor", "UK-AMo", "US-Me3", "DE-RuW", "DK-Vng", "US-GLE", "US-Tw4",
  "US-Sta", "CA-NS3", "US-Wi7", "US-YK2", "BE-Dor", "US-Lin", "US-KS3",
  "US-Ha1", "DE-Tha"
)

# Only keep those that actually exist in this ZIP
sites_with_warning <- intersect(sites_with_warning, unique_sites)
sites_with_warning
length(sites_with_warning)

[1] "NL-Loo" "US-Myb" "US-Wi9" "CA-NS1" "SE-Svb" "FR-Gri" "US-Tw1" "IT-Noe"
 [9] "CA-NS2" "CA-NS4" "FR-Fon" "US-KS2" "US-Oho" "BE-Vie" "ES-LMa" "US-Twt"
[17] "US-ORv" "US-Whs" "FI-Sod" "CA-TPD" "DE-RuS" "IT-BCi" "CA-Qfo" "US-RGo"
[25] "US-Wi0" "US-ARM" "CA-TP3" "AU-Cum" "SE-Nor" "UK-AMo" "US-Me3" "DE-RuW"
[33] "DK-Vng" "US-GLE" "US-Tw4" "US-Sta" "CA-NS3" "US-Wi7" "US-YK2" "BE-Dor"
[41] "US-Lin" "US-KS3" "US-Ha1" "DE-Tha"

[1] 44

In [7]:
# --- Cell 5: Function to process one site ---

process_one_site <- function(site) {
  message("=== Processing site: ", site, " ===")
  
  # Files inside the zip that correspond to this site
  site_files  <- csv_files[site_ids == site]
  merged_path <- file.path(outdir, paste0(site, "_merged.csv"))
  
  # If re-running, start from scratch
  if (file.exists(merged_path)) file.remove(merged_path)
  
  first_write <- TRUE
  
  for (f in site_files) {
    message("    reading file: ", f)
    dt <- read_file_from_zip(f)
    
    if (nrow(dt) == 0) {
      next
    }
    
    # Columns to keep: intersection of wanted + timestamps with what's available
    keep <- intersect(c("TIMESTAMP_START", "TIMESTAMP_END", wanted_cols),
                      names(dt))
    dt <- dt[, ..keep]
    
    # Append to per-site file
    fwrite(dt, merged_path, append = !first_write)
    first_write <- FALSE
    
    rm(dt); gc()
  }
  
  # If nothing was written (e.g. all files were empty / missing), skip
  if (!file.exists(merged_path)) {
    message("    no data written for site ", site)
    return(invisible(FALSE))
  }
  
  # Now read merged file and remove duplicates
  dt_all <- fread(merged_path, showProgress = FALSE)
  
  ts_cols <- intersect(c("TIMESTAMP_START", "TIMESTAMP_END"), names(dt_all))
  if (length(ts_cols) > 0L) {
    setkeyv(dt_all, ts_cols)
    dt_all <- unique(dt_all)
  } else {
    dt_all <- unique(dt_all)
  }
  
  fwrite(dt_all, merged_path)
  rm(dt_all); gc()
  
  invisible(TRUE)
}

In [8]:
# --- Cell 6: Run on warning sites first (parallel) ---

# Use separate R processes for safety in Jupyter
plan(multisession, workers = 3)   # adjust workers depending on RAM

# Optional: you can also test sequentially first:
# lapply(sites_with_warning, process_one_site)

res_warning <- future_lapply(sites_with_warning, process_one_site)

res_warning

=== Processing site: NL-Loo ===

    reading file: HH_flux/ICOSETC_NL-Loo_FLUXNET_HH_L2.csv

    reading file: __MACOSX/HH_flux/._ICOSETC_NL-Loo_FLUXNET_HH_L2.csv

    reading file: HH_flux/FLX_NL-Loo_FLUXNET2015_FULLSET_HH_1996-2018_beta-3.csv

    reading file: __MACOSX/HH_flux/._FLX_NL-Loo_FLUXNET2015_FULLSET_HH_1996-2018_beta-3.csv

    reading file: HH_flux/ICOSETC_NL-Loo_FLUXNET_HH_L2 copy.csv

    reading file: __MACOSX/HH_flux/._ICOSETC_NL-Loo_FLUXNET_HH_L2 copy.csv

Warning message in grepl("^Mac OS X", col1):
“unable to translate 'Mac OS X        	2~<b0>ATTR<b0><98><98>com.apple.quarantineq/0081' to a wide string”
Warning message in grepl("^Mac OS X", col1):
“input string 1 is invalid”
Warning message in grepl("com\\.apple\\.quarantine", col1):
“unable to translate 'Mac OS X        	2~<b0>ATTR<b0><98><98>com.apple.quarantineq/0081' to a wide string”
Warning message in grepl("com\\.apple\\.quarantine", col1):
“input string 1 is invalid”
Warning message in f

[[1]]
[1] TRUE

[[2]]
[1] TRUE

[[3]]
[1] TRUE

[[4]]
[1] TRUE

[[5]]
[1] TRUE

[[6]]
[1] TRUE

[[7]]
[1] TRUE

[[8]]
[1] TRUE

[[9]]
[1] TRUE

[[10]]
[1] TRUE

[[11]]
[1] TRUE

[[12]]
[1] TRUE

[[13]]
[1] TRUE

[[14]]
[1] TRUE

[[15]]
[1] TRUE

[[16]]
[1] TRUE

[[17]]
[1] TRUE

[[18]]
[1] TRUE

[[19]]
[1] TRUE

[[20]]
[1] TRUE

[[21]]
[1] TRUE

[[22]]
[1] TRUE

[[23]]
[1] TRUE

[[24]]
[1] TRUE

[[25]]
[1] TRUE

[[26]]
[1] TRUE

[[27]]
[1] TRUE

[[28]]
[1] TRUE

[[29]]
[1] TRUE

[[30]]
[1] TRUE

[[31]]
[1] TRUE

[[32]]
[1] TRUE

[[33]]
[1] TRUE

[[34]]
[1] TRUE

[[35]]
[1] TRUE

[[36]]
[1] TRUE

[[37]]
[1] TRUE

[[38]]
[1] TRUE

[[39]]
[1] TRUE

[[40]]
[1] TRUE

[[41]]
[1] TRUE

[[42]]
[1] TRUE

[[43]]
[1] TRUE

[[44]]
[1] TRUE

In [9]:
# --- Quick check of one processed site ---
test_site <- sites_with_warning[1]
test_path <- file.path(outdir, paste0(test_site, "_merged.csv"))
test_dt   <- fread(test_path, nrows = 10)
test_dt

TIMESTAMP_START,TIMESTAMP_END,TA_F,TA_F_QC,VPD_F,VPD_F_QC,P_F,P_F_QC,TS_F_MDS_1,TS_F_MDS_2,⋯,LW_OUT,SW_OUT,NETRAD,USTAR,WS,G_F_MDS,G_F_MDS_QC,RH,NIGHT,PA
<int64>,<int64>,<dbl>,<int>,<dbl>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
202301010000,202301010030,10.209,2,2.840,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999
202301010030,202301010100,10.165,2,2.760,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999
202301010100,202301010130,10.122,2,2.680,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999
202301010130,202301010200,10.053,2,2.646,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999
202301010200,202301010230,9.984,2,2.611,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999
202301010230,202301010300,10.059,2,2.602,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999
202301010300,202301010330,10.134,2,2.593,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999
202301010330,202301010400,10.210,2,2.596,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999
202301010400,202301010430,10.286,2,2.599,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999


In [19]:
### Let's check the table of sites with warning
test <- fread(file.path(outdir, "NL-Loo_merged.csv"))
head(test)
nrow(test)


TIMESTAMP_START,TIMESTAMP_END,TA_F,TA_F_QC,VPD_F,VPD_F_QC,P_F,P_F_QC,TS_F_MDS_1,TS_F_MDS_2,⋯,LW_OUT,SW_OUT,NETRAD,USTAR,WS,G_F_MDS,G_F_MDS_QC,RH,NIGHT,PA
<int64>,<int64>,<dbl>,<int>,<dbl>,<int>,<dbl>,<int>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>,<dbl>
202301010000,202301010030,10.209,2,2.840,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999
202301010030,202301010100,10.165,2,2.760,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999
202301010100,202301010130,10.122,2,2.680,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999
202301010130,202301010200,10.053,2,2.646,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999
202301010200,202301010230,9.984,2,2.611,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999
202301010230,202301010300,10.059,2,2.602,2,0,2,-9999,-9999,⋯,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,1,-9999


[1] 35088

In [ ]:
# --- Cell 7: Process all sites (full run) ---

sites_to_do <- unique_sites

# If you want to skip sites already done in this new folder:
existing   <- list.files(outdir, pattern = "_merged\\.csv$", full.names = FALSE)
sites_done <- str_extract(existing, "[A-Z]{2}-[A-Za-z0-9]+")
sites_to_do <- setdiff(unique_sites, sites_done)

length(sites_to_do)

plan(multisession, workers = 3)  # or 2 or 4, depending on htop

res_all <- future_lapply(sites_to_do, process_one_site)

res_all